# TimeDART: Parallel CNN Patch Embedding + Causal Adapter

GPU notebook for the original 144-recording experiment. Target: **CAUSAL_ADAPTER**; original BI_FULL and BI_ADAPTER comparisons are also retained.

Pretraining: causal Transformer + patch diffusion. Fine-tuning: frozen backbone + trainable 64→8→64 adapters and classification head.

Input: full 50,001-point A-scan → 20× FIR resampling → 2,501 points; patch/stride 32; CNN kernels 3 and 5; ordered time-bin pooling + global max.

The five outer folds rotate across all 144 recordings. This is internal cross-validation on previously explored data, not a new independent external test.

Private recording lists, waveform hashes, and earlier fold assignments are excluded. The notebook generates new folds locally from your uploaded dataset. It does not promise to reproduce an earlier AUROC.


In [ ]:
# GPU environment
import sys, subprocess, importlib.util, json, zipfile, datetime
from pathlib import Path

required = {"numpy": "numpy", "scipy": "scipy", "sklearn": "scikit-learn",
            "matplotlib": "matplotlib", "pandas": "pandas"}
missing = [package for module, package in required.items()
           if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", *missing])
import torch
if not torch.cuda.is_available():
    raise RuntimeError("Select a GPU runtime in Colab, then rerun this cell.")
DEVICE = "cuda"
print("PyTorch:", torch.__version__)
print("GPU:", torch.cuda.get_device_name(0))


In [ ]:
# Public source only: private recording metadata is generated at runtime
PROJECT = Path("/content/TimeDART_CNN_CausalAdapter")
PROJECT.mkdir(parents=True, exist_ok=True)
SOURCE_FILES = json.loads("{\"README_KO.md\":\"# 병렬 CNN 패치 임베딩 + Causal Adapter\\n\\n핵심 모델은 CAUSAL_ADAPTER입니다.\\n\\n- 전체 50,001-point 전압을 20배 FIR resampling하여 2,501 points로 입력합니다.\\n- patch/stride=32, CNN kernel 3·5 병렬 분기, hidden=64, causal Transformer 2층입니다.\\n- 사전학습은 shifted causal context + patch diffusion + self-only decoder입니다.\\n- Adapter는 사전학습에 넣지 않고 분류 파인튜닝에서 각 encoder 층 뒤에 추가합니다.\\n- Adapter: 64→8→64 residual MLP. Backbone을 고정하고 Adapter와 head만 학습합니다.\\n- 분류 head: 8개 시간 구간 평균 + global max pooling.\\n- 기존 비교군 BI_FULL, BI_ADAPTER도 보존했습니다.\\n\\n## 공개용 변경\\n데이터 파일명, 파형 해시, 기존 고정 fold 목록은 공개하지 않습니다.\\nprepare_local_metadata.py가 사용자가 넣은 ZIP에서 이 파일들을 로컬로 생성합니다.\\n새로 생성한 fold는 과거 실험의 저장된 fold와 동일하다고 보장하지 않습니다.\\n따라서 과거 AUROC를 그대로 재현한 배포판으로 주장하지 않습니다.\\n\\n## 실행\\npython prepare_local_metadata.py --data /path/to/data.zip\\npython checks.py\\npython cv.py train --data /path/to/data.zip --out /path/to/new_run --device cuda\\npython cv.py test --data /path/to/data.zip --out /path/to/new_run --device cuda\\n\\n원본의 model.py, data.py, run.py, cv.py, checks.py 및 config.json은 보존했습니다.\\n공식 TimeDART의 완전한 재현이 아닌 NDT 적용 변형입니다.\\n\",\"checks.py\":\"\\\"\\\"\\\"Check mask isolation, adapter gradients and actual frozen-weight invariance.\\\"\\\"\\\"\\nimport json,tempfile\\nfrom pathlib import Path\\nimport torch\\nfrom model import ModelConfig,Pretrainer,Classifier,VARIANTS\\n\\ndef run_checks(device='cpu'):\\n    torch.set_num_threads(2);torch.manual_seed(713)\\n    cfg=ModelConfig();n=(cfg.length+cfg.patch-1)//cfg.patch\\n    p=torch.randn(2,n,cfg.patch,device=device);q=p.clone();q[:,8:]+=7\\n    pre=Pretrainer(cfg,'cnn').to(device).eval()\\n    with torch.no_grad():\\n        a=pre.backbone.context_from_patches(p,True);b=pre.backbone.context_from_patches(q,True)\\n        assert torch.allclose(a[:,:9],b[:,:9],atol=1e-5),'Pretraining leaks future clean patches'\\n        assert torch.allclose(pre.backbone.embed(p)[:,:8],pre.backbone.embed(q)[:,:8],atol=1e-6)\\n        context=torch.randn(2,n,64,device=device);query=torch.randn_like(context)\\n        changed=context.clone();changed[:,8:]+=9\\n        assert torch.allclose(pre.decoder(query,context)[:,:8],pre.decoder(query,changed)[:,:8],atol=1e-6)\\n    pre.train();loss=pre.loss(torch.randn(2,2501,device=device)*.004)\\n    loss.backward();assert torch.isfinite(loss)\\n    assert pre.backbone.point_valid.sum()==2501\\n    results={};initial_head=None\\n    for variant in VARIANTS:\\n        torch.manual_seed(713)\\n        model=Classifier(cfg,variant).to(device).eval()\\n        model.backbone.load_state_dict(pre.backbone.state_dict())\\n        if initial_head is None:initial_head=model.head[-1].weight.detach().clone()\\n        else:assert torch.equal(initial_head,model.head[-1].weight),'Unpaired head initialization'\\n        with torch.no_grad():\\n            a=model.encode_patches(p);b=model.encode_patches(q)\\n            if model.bidirectional:\\n                assert not torch.allclose(a[:,:8],b[:,:8],atol=1e-5),'Bidirectional path cannot see future'\\n            else:\\n                assert torch.allclose(a[:,:8],b[:,:8],atol=1e-5),'Causal classifier leaks future'\\n            # Initially zero-up adapters must not change the original encoder function.\\n            mask=None if model.bidirectional else model.backbone.causal_mask\\n            reference=model.backbone.encoder(model.backbone.embed(p)+model.backbone.pe,mask=mask)\\n            assert torch.allclose(a,reference,atol=1e-5),'Initial adapters are not identity'\\n        # Pooling against a sample-level reference, excluding padded duration.\\n        z=torch.randn(2,n,64,device=device)\\n        expanded=z.repeat_interleave(32,dim=1)[:,:2501];edges=[i*2501//8 for i in range(9)]\\n        expected=torch.cat([torch.stack([expanded[:,edges[i]:edges[i+1]].mean(1) for i in range(8)],1).flatten(1),z.amax(1)],1)\\n        assert torch.allclose(model.pool(z),expected,atol=1e-6)\\n        before={k:v.detach().clone() for k,v in model.backbone.state_dict().items()}\\n        trainable=[p for p in model.parameters() if p.requires_grad]\\n        if model.use_adapter:\\n            assert sum(p.numel() for p in trainable)==2769\\n            assert all(n.startswith(('adapters.','head.')) for n,p in model.named_parameters() if p.requires_grad)\\n        opt=torch.optim.AdamW(trainable,lr=.001)\\n        model.train();x=torch.randn(4,2501,device=device)*.004;y=torch.tensor([0.,1.,0.,1.],device=device)\\n        for _ in range(2):\\n            opt.zero_grad(set_to_none=True)\\n            loss=torch.nn.functional.binary_cross_entropy_with_logits(model(x),y)\\n            assert torch.isfinite(loss);loss.backward()\\n            if model.use_adapter:\\n                assert all(p.grad is None for p in model.backbone.parameters())\\n                assert all(a.up.weight.grad is not None and a.up.weight.grad.abs().sum()>0 for a in model.adapters)\\n            opt.step()\\n        if model.use_adapter:\\n            assert all(torch.equal(before[k],v) for k,v in model.backbone.state_dict().items()),'Frozen backbone changed'\\n            assert all(a.down.weight.grad is not None and a.down.weight.grad.abs().sum()>0 for a in model.adapters),'Gradient lost through later frozen layer'\\n        else:\\n            assert not torch.equal(before['embed.proj.weight'],model.backbone.embed.proj.weight),'Full tuning did not update embedding'\\n        model.eval()\\n        with tempfile.TemporaryDirectory() as td:\\n            path=Path(td)/'check.pt';torch.save(model.state_dict(),path)\\n            restored=Classifier(cfg,variant).to(device).eval()\\n            restored.load_state_dict(torch.load(path,map_location=device,weights_only=True))\\n            with torch.no_grad():assert torch.allclose(model(x),restored(x),atol=1e-6)\\n        results[variant]={'parameters':sum(p.numel() for p in model.parameters()),\\n                          'trainable_parameters':sum(p.numel() for p in trainable),\\n                          'classification_attention':'bidirectional' if model.bidirectional else 'causal',\\n                          'pretraining':'causal, self-only decoder, no adapters',\\n                          'mask_pool_gradient_freeze_checkpoint_checks':'passed'}\\n    print(json.dumps(results,indent=2));return results\\n\\nif __name__=='__main__':run_checks('cuda' if torch.cuda.is_available() else 'cpu')\\n\",\"config.json\":\"{\\n  \\\"seed\\\": 42,\\n  \\\"factor\\\": 20,\\n  \\\"dim\\\": 64,\\n  \\\"layers\\\": 2,\\n  \\\"heads\\\": 4,\\n  \\\"ff\\\": 128,\\n  \\\"dropout\\\": 0.1,\\n  \\\"batch_size\\\": 16,\\n  \\\"pretrain_epochs\\\": 60,\\n  \\\"pretrain_patience\\\": 15,\\n  \\\"finetune_epochs\\\": 150,\\n  \\\"finetune_patience\\\": 25,\\n  \\\"pretrain_lr\\\": 0.0005,\\n  \\\"encoder_lr\\\": 0.0001,\\n  \\\"head_lr\\\": 0.001,\\n  \\\"weight_decay\\\": 0.001,\\n  \\\"patch\\\": 32,\\n  \\\"adapter_dim\\\": 8,\\n  \\\"adapter_lr\\\": 0.001\\n}\",\"cv.py\":\"\\\"\\\"\\\"Prespecified 5-fold outer evaluation with one stratified inner holdout.\\\"\\\"\\\"\\nimport argparse, csv, json, shutil\\nfrom collections import Counter\\nfrom pathlib import Path\\nimport numpy as np\\nimport torch\\nfrom data import manifest\\nfrom run import ROOT, VARIANTS, dump, read, sha, writecsv, compare, final_test, device_for\\n\\nSOURCES=('model.py','data.py','run.py','cv.py','dataset_manifest.json','fixed_folds.json','checks.py')\\nMETRICS=('auroc','average_precision','balanced_accuracy','f1','recall','specificity')\\n\\ndef make_splits():\\n    # Freeze the exact folds from the user's completed A/B experiment.\\n    docs=read(ROOT/'fixed_folds.json');assert len(docs)==5\\n    reference={r['file'] for r in read(ROOT/'dataset_manifest.json')['records']}\\n    all_test=[]\\n    for fold,doc in enumerate(docs,1):\\n        assert doc['fold']==fold\\n        roles={k:{r['file'] for r in v} for k,v in doc['splits'].items()}\\n        assert not (roles['train']&roles['validation'] or roles['train']&roles['test'] or roles['validation']&roles['test'])\\n        assert set.union(*roles.values())==reference\\n        assert all({r['label'] for r in rows}=={0,1} for rows in doc['splits'].values())\\n        all_test.extend(roles['test'])\\n    assert Counter(all_test)==Counter(reference)\\n    return docs\\n\\ndef prepare(data_path,root,c):\\n    root=Path(root);root.mkdir(parents=True,exist_ok=True)\\n    if c.get('patch')!=32 or c.get('adapter_dim')!=8:raise ValueError('This comparison fixes patch32 and adapter bottleneck8')\\n    docs=make_splits()\\n    evidence={'config':c,'n_splits':5,'outer_seed':42,'inner_val_fraction':.20,\\n              'inner_seeds':[1001,1002,1003,1004,1005],\\n              'training_seed_rule':'same config.seed for all folds; fresh initialization in each fold',\\n              'source_sha256':{n:sha(ROOT/n) for n in SOURCES},'folds':docs}\\n    guard=root/'cv_protocol.json'\\n    if guard.exists() and read(guard)!=evidence:\\n        raise ValueError('Code/config/folds changed. Use a new RUN_NAME; do not reuse old checkpoints.')\\n    dump(guard,evidence);splitdir=root/'splits';splitdir.mkdir(exist_ok=True)\\n    counts=[]\\n    for doc in docs:\\n        fold=doc['fold'];path=splitdir/f'fold_{fold:02d}.json'\\n        if path.exists() and read(path)!=doc:raise ValueError('Saved fold manifest changed')\\n        dump(path,doc);manifest(data_path,path)\\n        for role,rows in doc['splits'].items():\\n            counts.append({'fold':fold,'role':role,'n':len(rows),\\n                           'normal':sum(r['label']==0 for r in rows),'defect':sum(r['label']==1 for r in rows)})\\n    writecsv(root/'fold_counts.csv',counts)\\n    snapshot=root/'source_snapshot';snapshot.mkdir(exist_ok=True)\\n    for n in (*SOURCES,'README_KO.md'):\\n        shutil.copy2(ROOT/n,snapshot/n)\\n    dump(snapshot/'config.json',c)\\n    return docs\\n\\ndef verify_run(data_path,root):\\n    p=read(root/'cv_protocol.json')\\n    for n,h in p['source_sha256'].items():\\n        if sha(ROOT/n)!=h:raise ValueError('Code changed: '+n)\\n    for doc in p['folds']:\\n        split=root/'splits'/f\\\"fold_{doc['fold']:02d}.json\\\"\\n        if read(split)!=doc:raise ValueError('Fold changed')\\n        manifest(data_path,split)\\n        folder=root/f\\\"fold_{doc['fold']:02d}\\\"\\n        protocol=read(folder/'protocol.json')\\n        if protocol['config']!=p['config'] or protocol['split_sha256']!=sha(split):\\n            raise ValueError('Training protocol mismatch')\\n        selection=read(folder/'selection_before_test.json')\\n        for v in VARIANTS:\\n            if sha(folder/v/'best.pt')!=selection['checkpoint_sha256'][v]:raise ValueError('Checkpoint changed')\\n            if sha(folder/v/'validation.json')!=selection['validation_file_sha256'][v]:raise ValueError('Validation changed')\\n    return p\\n\\ndef summarize(root):\\n    rows=[];selected=[]\\n    for fold in range(1,6):\\n        folder=root/f'fold_{fold:02d}'\\n        test_rows=list(csv.DictReader((folder/'test_comparison.csv').open(encoding='utf-8-sig')))\\n        if {r['variant'] for r in test_rows}!=set(VARIANTS) or len(test_rows)!=len(VARIANTS):raise ValueError('Missing model result')\\n        winner=read(folder/'selection_before_test.json')['selected_by_validation']\\n        for r in test_rows:\\n            r={k:(float(v) if k not in ('variant','selected_by_validation') else v) for k,v in r.items()}\\n            r['fold']=fold;r['selected_by_validation']=r['variant']==winner\\n            rows.append(r)\\n            if r['variant']==winner:selected.append(dict(r))\\n    writecsv(root/'cv_fold_metrics.csv',rows)\\n    writecsv(root/'cv_inner_selected_fold_metrics.csv',selected)\\n    def aggregate(group,name):\\n        out={'variant':name,'n_folds':len(group)}\\n        for metric in METRICS:\\n            a=np.array([r[metric] for r in group])\\n            out[metric+'_mean']=float(a.mean());out[metric+'_std']=float(a.std(ddof=1))\\n        return out\\n    summary=[aggregate([r for r in rows if r['variant']==v],v) for v in VARIANTS]\\n    writecsv(root/'cv_summary.csv',summary)\\n    writecsv(root/'cv_inner_selected_summary.csv',[aggregate(selected,'inner_validation_selection')])\\n    import matplotlib\\n    matplotlib.use('Agg')\\n    import matplotlib.pyplot as plt\\n    fig,ax=plt.subplots(figsize=(7,4.5))\\n    for v in VARIANTS:\\n        group=[r for r in rows if r['variant']==v]\\n        ax.plot([r['fold'] for r in group],[r['auroc'] for r in group],marker='o',label=v)\\n    ax.axhline(.5,color='gray',linestyle='--',linewidth=1)\\n    ax.axhline(.85,color='gray',linestyle=':',linewidth=1)\\n    ax.set(xlabel='Outer test fold',ylabel='AUROC',ylim=(0,1),xticks=range(1,6),title='Prespecified models: stratified 5-fold')\\n    ax.legend();fig.tight_layout();fig.savefig(root/'cv_auroc.png',dpi=160);plt.close(fig)\\n    dump(root/'cv_notes.json',{\\n        'summary':'Unweighted mean and sample standard deviation (ddof=1) across five outer folds. SD is not a confidence interval.',\\n        'selected_policy':'Separate summary for a policy choosing among CAUSAL_ADAPTER/BI_FULL/BI_ADAPTER using only the inner validation AUROC (tie: validation loss).',\\n        'limitations':'Only 18 defects; each outer test has 3 or 4. Overlapping training sets make folds dependent. All records were previously explored; internal resampling, not an untouched external test.',\\n        'groups':'Stratification is by label, not specimen/session/location. This does not establish generalization to new specimens or sessions.',\\n        'selection':'Architectures/config are fixed before running. Do not select settings using outer scores and call those scores unbiased performance.',\\n        'predictions':'Per-fold prediction files are retained for audit. No pooled OOF score or pooled OOF AUROC is computed.',\\n        'target':'AUROC >= 0.85 is not guaranteed.'})\\n    print(json.dumps(summary,indent=2),flush=True)\\n    return summary\\n\\ndef main():\\n    ap=argparse.ArgumentParser()\\n    ap.add_argument('stage',choices=('prepare','train','test','summary'))\\n    ap.add_argument('--data',required=True);ap.add_argument('--out',required=True)\\n    ap.add_argument('--config',default=str(ROOT/'config.json'))\\n    ap.add_argument('--device',choices=('cpu','cuda'),default='cuda')\\n    ap.add_argument('--threads',type=int,default=2)\\n    a=ap.parse_args();root=Path(a.out);torch.set_num_threads(a.threads)\\n    if a.stage in ('prepare','train'):\\n        c=read(a.config);prepare(a.data,root,c)\\n        if a.stage=='prepare':return\\n        device=device_for(a.device)\\n        for fold in range(1,6):\\n            print(f'\\\\n===== TRAIN FOLD {fold}/5 =====',flush=True)\\n            compare(a.data,root/'splits'/f'fold_{fold:02d}.json',root/f'fold_{fold:02d}',c,device)\\n    else:\\n        verify_run(a.data,root)\\n        if a.stage=='test':\\n            device=device_for(a.device)\\n            for fold in range(1,6):\\n                print(f'\\\\n===== TEST FOLD {fold}/5 =====',flush=True)\\n                final_test(a.data,root/'splits'/f'fold_{fold:02d}.json',root/f'fold_{fold:02d}',device)\\n        summarize(root)\\n\\nif __name__=='__main__':main()\\n\",\"data.py\":\"\\\"\\\"\\\"Scan-level folds; exact 144-record identity and split isolation checks.\\\"\\\"\\\"\\nimport hashlib,io,json,zipfile\\nfrom pathlib import Path\\nimport numpy as np\\nfrom scipy.signal import resample_poly\\n\\nROLES=('train','validation','test')\\n\\ndef canonical(n):\\n    n=n.replace('\\\\\\\\','/')\\n    hits=[n.find(v) for v in ('##정상/','##결함/') if v in n]\\n    if len(hits)!=1:raise ValueError('Unknown class folder: '+n)\\n    return n[hits[0]:]\\n\\ndef manifest(data_path,split_path):\\n    doc=json.loads(Path(split_path).read_text(encoding='utf-8'))\\n    allrows=[];seen_hash=set()\\n    if set(doc['splits'])!=set(ROLES):raise ValueError('Missing/extra split roles')\\n    for role in ROLES:\\n        rows=doc['splits'][role]\\n        if not rows or any(sum(r['label']==v for r in rows)==0 for v in (0,1)):raise ValueError('Each role requires both classes')\\n        for r in rows:\\n            if r['label']!=int('##결함/' in r['file']):raise ValueError('Label mismatch')\\n            if r['waveform_sha256'] in seen_hash:raise ValueError('Duplicate signal across/within splits')\\n            seen_hash.add(r['waveform_sha256'])\\n        allrows+=rows\\n    reference=json.loads((Path(__file__).parent/'dataset_manifest.json').read_text(encoding='utf-8'))['records']\\n    if sorted(allrows,key=lambda r:r['file'])!=reference:raise ValueError('Fold must contain exactly the known 144 unique records with original labels/hashes')\\n    with zipfile.ZipFile(data_path) as z:\\n        names=[n for n in z.namelist() if n.lower().endswith('.dat') and not n.startswith('__MACOSX/')]\\n    lookup={canonical(n):n for n in names}\\n    expected={r['file'] for r in allrows}\\n    if len(names)!=144 or len(lookup)!=144 or set(lookup)!=expected:raise ValueError('This package requires the exact 144-file data(5)(1).zip / equivalent data(5).zip, not the 274-file dataset.')\\n    return doc,lookup\\n\\ndef load_roles(data_path,split_path,roles,factor=20):\\n    if not isinstance(factor,int) or factor<1:raise ValueError('factor must be positive integer')\\n    doc,lookup=manifest(data_path,split_path);out={};reference=None\\n    with zipfile.ZipFile(data_path) as z:\\n        for role in roles:\\n            if role not in ROLES:raise ValueError(role)\\n            xs=[];ys=[];names=[]\\n            for r in doc['splits'][role]:\\n                a=np.loadtxt(io.BytesIO(z.read(lookup[r['file']])))\\n                if a.shape==(50001,2):a=a.T\\n                if a.shape!=(2,50001) or not np.isfinite(a).all():raise ValueError('Invalid shape/nonfinite signal: '+r['file'])\\n                t,v=a\\n                if not np.all(np.diff(t)>0):raise ValueError('Invalid time axis')\\n                if reference is None:reference=t\\n                if not np.allclose(t,reference,rtol=0,atol=1e-15):raise ValueError('Time axes differ')\\n                v=v.astype(np.float32)\\n                if hashlib.sha256(v.tobytes()).hexdigest()!=r['waveform_sha256']:raise ValueError('Signal hash does not match fixed dataset: '+r['file'])\\n                # Match the attached visualization notebook: float32 -> float64 -> resample_poly.\\n                q=resample_poly(v.astype(np.float64),1,factor) if factor!=1 else v\\n                xs.append(q.astype(np.float32));ys.append(r['label']);names.append(r['file'])\\n            out[role]=(np.stack(xs),np.array(ys,np.int64),names)\\n    return out\\n\",\"model.py\":\"\\\"\\\"\\\"Compact paper-based TimeDART adaptation, not a byte-for-byte official port.\\nReference: https://arxiv.org/html/2410.05711v3 (sections 3.1-3.3).\\n\\\"\\\"\\\"\\nimport math\\nfrom dataclasses import dataclass, asdict\\nimport torch\\nfrom torch import nn\\nfrom torch.nn import functional as F\\n\\n@dataclass\\nclass ModelConfig:\\n    length: int = 2501\\n    patch: int = 32\\n    dim: int = 64\\n    layers: int = 2\\n    heads: int = 4\\n    ff: int = 128\\n    dropout: float = 0.1\\n    diffusion_steps: int = 1000\\n    adapter_dim: int = 8\\n\\nclass PatchEmbedding(nn.Module):\\n    def __init__(self, cfg, kind):\\n        super().__init__(); self.kind=kind; self.cfg=cfg\\n        if kind=='linear':\\n            self.proj=nn.Linear(cfg.patch,cfg.dim)\\n        elif kind=='cnn':\\n            self.conv3=nn.Conv1d(1,8,3,padding=1)\\n            self.conv5=nn.Conv1d(1,8,5,padding=2)\\n            self.proj=nn.Linear(16*cfg.patch,cfg.dim)\\n        else: raise ValueError(kind)\\n    def forward(self,p):\\n        # Each B*N patch becomes a separate batch item: convolutions cannot cross patches.\\n        if self.kind=='linear': return self.proj(p)\\n        b,n,l=p.shape; q=p.reshape(b*n,1,l)\\n        q=torch.cat([F.gelu(self.conv3(q)),F.gelu(self.conv5(q))],dim=1)\\n        return self.proj(q.flatten(1)).reshape(b,n,-1)\\n\\nclass Backbone(nn.Module):\\n    def __init__(self,cfg,kind):\\n        super().__init__(); self.cfg=cfg; self.kind=kind\\n        # Construct common modules before variant-specific embedding for paired initialization.\\n        layer=nn.TransformerEncoderLayer(cfg.dim,cfg.heads,cfg.ff,cfg.dropout,\\n                                         activation='gelu',batch_first=True,norm_first=True)\\n        self.encoder=nn.TransformerEncoder(layer,cfg.layers,norm=nn.LayerNorm(cfg.dim),enable_nested_tensor=False)\\n        self.sos=nn.Parameter(torch.zeros(1,1,cfg.dim));nn.init.normal_(self.sos,std=.02)\\n        self.embed=PatchEmbedding(cfg,kind)\\n        n=math.ceil(cfg.length/cfg.patch)\\n        pos=torch.arange(n,dtype=torch.float32).unsqueeze(1)\\n        div=torch.exp(torch.arange(0,cfg.dim,2,dtype=torch.float32)*(-math.log(10000)/cfg.dim))\\n        pe=torch.zeros(n,cfg.dim);pe[:,0::2]=torch.sin(pos*div);pe[:,1::2]=torch.cos(pos*div)\\n        self.register_buffer('pe',pe.unsqueeze(0))\\n        self.register_buffer('causal_mask',torch.ones(n,n,dtype=torch.bool).triu(1))\\n        weights=torch.ones(n*cfg.patch);weights[cfg.length:]=0\\n        self.register_buffer('point_valid',weights.reshape(n,cfg.patch))\\n    def patchify(self,x):\\n        if x.ndim!=2 or x.shape[1]!=self.cfg.length:raise ValueError(f'Expected [B,{self.cfg.length}], got {tuple(x.shape)}')\\n        # Paper-style per-scan instance normalization; never fitted across records.\\n        mu=x.mean(1,keepdim=True);sd=x.std(1,keepdim=True,unbiased=False).clamp_min(1e-8)\\n        q=(x-mu)/sd\\n        q=F.pad(q,(0,(-q.shape[1])%self.cfg.patch),mode='replicate')\\n        return q.reshape(len(x),-1,self.cfg.patch)\\n    def context_from_patches(self,p,shifted):\\n        z=self.embed(p)\\n        if shifted:z=torch.cat([self.sos.expand(len(p),-1,-1),z[:,:-1]],dim=1)\\n        return self.encoder(z+self.pe,mask=self.causal_mask)\\n    def forward(self,x):return self.context_from_patches(self.patchify(x),shifted=False)\\n\\nclass Denoiser(nn.Module):\\n    def __init__(self,cfg):\\n        super().__init__()\\n        self.cross=nn.MultiheadAttention(cfg.dim,cfg.heads,dropout=cfg.dropout,batch_first=True)\\n        self.n1=nn.LayerNorm(cfg.dim);self.n2=nn.LayerNorm(cfg.dim)\\n        self.ff=nn.Sequential(nn.Linear(cfg.dim,cfg.ff),nn.GELU(),nn.Dropout(cfg.dropout),nn.Linear(cfg.ff,cfg.dim))\\n        self.drop=nn.Dropout(cfg.dropout);self.out=nn.Linear(cfg.dim,cfg.patch)\\n    def forward(self,q,context):\\n        b,n,d=q.shape\\n        # Length-one attention per patch is algebraically diagonal/self-only cross-attention.\\n        q=q.reshape(b*n,1,d);c=context.reshape(b*n,1,d)\\n        a,_=self.cross(q,c,c,need_weights=False)\\n        h=self.n1(q+self.drop(a));h=self.n2(h+self.drop(self.ff(h)))\\n        return self.out(h).reshape(b,n,-1)\\n\\nclass Pretrainer(nn.Module):\\n    def __init__(self,cfg,kind):\\n        super().__init__();self.cfg=cfg;self.backbone=Backbone(cfg,kind);self.decoder=Denoiser(cfg)\\n        t=torch.linspace(0,cfg.diffusion_steps,cfg.diffusion_steps+1,dtype=torch.float64)\\n        abar=torch.cos(((t/cfg.diffusion_steps+.008)/1.008)*math.pi/2)**2\\n        abar=abar/abar[0];betas=(1-abar[1:]/abar[:-1]).clamp(0,.999)\\n        self.register_buffer('abar',torch.cumprod(1-betas,0).float())\\n    def loss(self,x,generator=None):\\n        clean=self.backbone.patchify(x)\\n        ctx=self.backbone.context_from_patches(clean,shifted=True)\\n        steps=torch.randint(self.cfg.diffusion_steps,clean.shape[:2],device=x.device,generator=generator)\\n        eps=torch.randn(clean.shape,device=x.device,dtype=clean.dtype,generator=generator)\\n        a=self.abar[steps].unsqueeze(-1)\\n        noisy=a.sqrt()*clean+(1-a).sqrt()*eps\\n        query=self.backbone.embed(noisy)+self.backbone.pe\\n        pred=self.decoder(query,ctx)\\n        valid=self.backbone.point_valid\\n        return ((pred-clean).square()*valid).sum()/(len(x)*valid.sum())\\n\\nVARIANTS=('CAUSAL_ADAPTER','BI_FULL','BI_ADAPTER')\\n\\nclass Adapter(nn.Module):\\n    def __init__(self,dim,bottleneck):\\n        super().__init__()\\n        self.down=nn.Linear(dim,bottleneck)\\n        self.up=nn.Linear(bottleneck,dim)\\n        nn.init.zeros_(self.up.weight);nn.init.zeros_(self.up.bias)\\n    def forward(self,h):return h+self.up(F.gelu(self.down(h)))\\n\\nclass Classifier(nn.Module):\\n    \\\"\\\"\\\"Eight ordered time-bin means + global max, followed by one linear head.\\\"\\\"\\\"\\n    def __init__(self,cfg,variant,amp_mean=None,amp_std=None):\\n        super().__init__();self.variant=variant;self.cfg=cfg\\n        if variant not in VARIANTS:raise ValueError(variant)\\n        self.use_adapter=variant!='BI_FULL'\\n        self.bidirectional=variant!='CAUSAL_ADAPTER'\\n        self.backbone=Backbone(cfg,'cnn')\\n        self.head=nn.Sequential(nn.Dropout(cfg.dropout),nn.Linear(9*cfg.dim,1))\\n        # Create the identical head before adapter initialization for matched head seeds.\\n        self.adapters=nn.ModuleList([Adapter(cfg.dim,cfg.adapter_dim) for _ in range(cfg.layers)] if self.use_adapter else [])\\n        if self.use_adapter:\\n            self.backbone.requires_grad_(False)\\n        # Weight each token by its REAL sample overlap with each time bin.\\n        # Padding contributes no additional duration. The partly valid last token remains.\\n        n=math.ceil(cfg.length/cfg.patch)\\n        edges=torch.arange(9,dtype=torch.int64)*cfg.length//8\\n        start=torch.arange(n,dtype=torch.int64)*cfg.patch\\n        end=(start+cfg.patch).clamp(max=cfg.length)\\n        overlap=(torch.minimum(edges[1:,None],end[None,:])-torch.maximum(edges[:-1,None],start[None,:])).clamp_min(0).float()\\n        if (overlap.sum(1)==0).any():raise ValueError('Input length must be at least 8')\\n        self.register_buffer('bin_weights',overlap/overlap.sum(1,keepdim=True))\\n        self.register_buffer('token_valid',end>start)\\n    def pool(self,z):\\n        regional=torch.einsum('bnd,kn->bkd',z,self.bin_weights).flatten(1)\\n        global_max=z.masked_fill(~self.token_valid[None,:,None],float('-inf')).amax(1)\\n        return torch.cat([regional,global_max],dim=1)\\n    def encode_patches(self,p):\\n        # Classification only: UNshifted input. Pretrainer continues using the original\\n        # shifted causal Backbone path, with no adapters and no mask removal.\\n        z=self.backbone.embed(p)+self.backbone.pe\\n        mask=None if self.bidirectional else self.backbone.causal_mask\\n        for i,layer in enumerate(self.backbone.encoder.layers):\\n            # Never use no_grad here: adapter gradients must pass through later frozen layers.\\n            z=layer(z,src_mask=mask)\\n            if self.use_adapter:z=self.adapters[i](z)\\n        return self.backbone.encoder.norm(z)\\n    def forward(self,x):\\n        z=self.encode_patches(self.backbone.patchify(x))\\n        return self.head(self.pool(z)).squeeze(-1)\\n\",\"run.py\":\"\\\"\\\"\\\"Three A3 extensions sharing the same causal pretraining within each fold.\\\"\\\"\\\"\\nimport os\\nos.environ.setdefault('CUBLAS_WORKSPACE_CONFIG',':4096:8')\\nimport argparse,csv,hashlib,json,math,random,sys,time\\nfrom pathlib import Path\\nfrom dataclasses import asdict\\nimport numpy as np\\nimport torch\\nfrom torch import nn\\nfrom torch.utils.data import DataLoader,TensorDataset\\nfrom scipy.special import expit\\nfrom sklearn.metrics import roc_auc_score,average_precision_score,balanced_accuracy_score,f1_score,confusion_matrix,roc_curve\\nfrom model import ModelConfig,Pretrainer,Classifier,VARIANTS\\nfrom data import load_roles,manifest\\n\\nROOT=Path(__file__).resolve().parent\\ndef dump(path,obj):\\n    path=Path(path);path.parent.mkdir(parents=True,exist_ok=True)\\n    tmp=path.with_suffix(path.suffix+'.tmp');tmp.write_text(json.dumps(obj,ensure_ascii=False,indent=2,allow_nan=False));os.replace(tmp,path)\\ndef read(path):return json.loads(Path(path).read_text())\\ndef sha(path):return hashlib.sha256(Path(path).read_bytes()).hexdigest()\\ndef savept(path,obj):\\n    path=Path(path);tmp=path.with_suffix('.tmp');torch.save(obj,tmp);os.replace(tmp,path)\\ndef loadpt(path,device='cpu'):return torch.load(path,map_location=device,weights_only=True)\\ndef seed_all(seed):\\n    random.seed(seed);np.random.seed(seed);torch.manual_seed(seed)\\n    if torch.cuda.is_available():torch.cuda.manual_seed_all(seed)\\n    torch.backends.cudnn.benchmark=False;torch.backends.cudnn.deterministic=True\\n    torch.use_deterministic_algorithms(True,warn_only=True)\\ndef device_for(name):\\n    if name=='cuda' and not torch.cuda.is_available():raise RuntimeError('CUDA GPU가 없습니다. Colab 런타임 유형을 T4 GPU로 바꾸세요.')\\n    return torch.device(name)\\ndef loader(arr,batch,shuffle=False,seed=42):\\n    x,y,_=arr\\n    ds=TensorDataset(torch.from_numpy(x),torch.from_numpy(y).float())\\n    return DataLoader(ds,batch_size=batch,shuffle=shuffle,num_workers=0,\\n                      generator=torch.Generator().manual_seed(seed),drop_last=False)\\ndef writecsv(path,rows):\\n    with Path(path).open('w',encoding='utf-8-sig',newline='') as f:\\n        w=csv.DictWriter(f,fieldnames=list(rows[0]));w.writeheader();w.writerows(rows)\\ndef predictions(path,arr,score,threshold):\\n    writecsv(path,[dict(file=n,label=int(y),defect_score=float(s),predicted=int(s>=threshold)) for n,y,s in zip(arr[2],arr[1],score)])\\ndef metrics(y,score,threshold=.5):\\n    pred=score>=threshold;tn,fp,fn,tp=confusion_matrix(y,pred,labels=[0,1]).ravel()\\n    return dict(auroc=float(roc_auc_score(y,score)),average_precision=float(average_precision_score(y,score)),\\n                balanced_accuracy=float(balanced_accuracy_score(y,pred)),f1=float(f1_score(y,pred,zero_division=0)),\\n                recall=float(tp/max(tp+fn,1)),specificity=float(tn/max(tn+fp,1)),tn=int(tn),fp=int(fp),fn=int(fn),tp=int(tp),threshold=float(threshold))\\ndef choose_threshold(y,p):\\n    u=np.unique(p);candidates=np.r_[0.,(u[:-1]+u[1:])/2,np.nextafter(u[-1],1.),.5]\\n    return float(max(candidates,key=lambda t:(balanced_accuracy_score(y,p>=t),-abs(t-.5))))\\n@torch.no_grad()\\ndef infer(model,arr,batch,device,criterion=None):\\n    model.eval();scores=[];loss=0.;n=0\\n    for x,y in loader(arr,batch):\\n        x=x.to(device);y=y.to(device);logit=model(x)\\n        if not torch.isfinite(logit).all():raise RuntimeError('Non-finite classification output')\\n        scores.extend(logit.sigmoid().cpu().numpy().tolist())\\n        if criterion is not None:loss+=float(criterion(logit,y))*len(x)\\n        n+=len(x)\\n    return np.array(scores),loss/max(n,1)\\n@torch.no_grad()\\ndef pre_val(model,arr,batch,device):\\n    model.eval();total=0.;n=0\\n    generator=torch.Generator(device=device).manual_seed(987654)\\n    for x,_ in loader(arr,batch):\\n        x=x.to(device);v=model.loss(x,generator)\\n        total+=float(v)*len(x);n+=len(x)\\n    return total/n\\ndef pretrain(kind,cfg,c,arrays,root,device):\\n    folder=root/('pretrain_'+kind);folder.mkdir(exist_ok=True)\\n    bestpath=folder/'best.pt';lastpath=folder/'last.pt';done=folder/'complete.json'\\n    if done.exists():print('Reuse completed pretraining:',kind,flush=True);return bestpath\\n    seed_all(c['seed']);model=Pretrainer(cfg,kind).to(device)\\n    opt=torch.optim.AdamW(model.parameters(),lr=c['pretrain_lr'],weight_decay=c['weight_decay'])\\n    history=[];best=float('inf');bad=0;start=1\\n    if lastpath.exists():\\n        state=loadpt(lastpath,device);model.load_state_dict(state['model']);opt.load_state_dict(state['optimizer'])\\n        history=state['history'];best=state['best'];bad=state['bad'];start=state['epoch']+1\\n        print('Resume pretrain',kind,'epoch',start,flush=True)\\n    if c['pretrain_epochs']==0:\\n        savept(bestpath,{'backbone':model.backbone.state_dict(),'epoch':0,'cfg':asdict(cfg)})\\n        dump(done,{'epoch':0,'note':'Random initialization; no pretraining'});return bestpath\\n    for epoch in range(start,c['pretrain_epochs']+1):\\n        if bad>=c['pretrain_patience']:break\\n        seed_all(c['seed']+epoch);model.train();total=0.;n=0;tick=time.time()\\n        for x,_ in loader(arrays['train'],c['batch_size'],True,c['seed']+epoch):\\n            x=x.to(device);opt.zero_grad(set_to_none=True);loss=model.loss(x)\\n            if not torch.isfinite(loss):raise RuntimeError('Non-finite pretraining loss')\\n            loss.backward();nn.utils.clip_grad_norm_(model.parameters(),1.);opt.step()\\n            total+=float(loss.detach())*len(x);n+=len(x)\\n        val=pre_val(model,arrays['validation'],c['batch_size'],device)\\n        if not math.isfinite(val):raise RuntimeError('Non-finite pretraining validation loss')\\n        if val<best-1e-7:\\n            best=val;bad=0;savept(bestpath,{'backbone':model.backbone.state_dict(),'epoch':epoch,'cfg':asdict(cfg),'val_loss':val})\\n        else:bad+=1\\n        history.append(dict(epoch=epoch,train_loss=total/n,val_loss=val))\\n        savept(lastpath,dict(model=model.state_dict(),optimizer=opt.state_dict(),history=history,best=best,bad=bad,epoch=epoch))\\n        writecsv(folder/'history.csv',history)\\n        print(f'PRE {kind:6s} {epoch:03d} train={total/n:.5f} val={val:.5f} best={best:.5f} {time.time()-tick:.1f}s',flush=True)\\n    dump(done,{'best_val_loss':best,'epochs_run':len(history),'checkpoint':str(bestpath)})\\n    del model,opt\\n    if device.type=='cuda':torch.cuda.empty_cache()\\n    return bestpath\\n\\ndef backbone_digest(model):\\n    h=hashlib.sha256()\\n    for n,t in sorted(model.backbone.state_dict().items()):\\n        h.update(n.encode());h.update(t.detach().cpu().contiguous().numpy().tobytes())\\n    return h.hexdigest()\\n\\ndef finetune(variant,prepath,cfg,c,arrays,root,device):\\n    folder=root/variant;folder.mkdir(exist_ok=True);summarypath=folder/'validation.json'\\n    if summarypath.exists():print('Reuse completed fine-tuning:',variant,flush=True);return read(summarypath)\\n    seed_all(c['seed']+10000);model=Classifier(cfg,variant).to(device)\\n    model.backbone.load_state_dict(loadpt(prepath,device)['backbone'])\\n    frozen_before=backbone_digest(model) if model.use_adapter else None\\n    profile={'variant':variant,'pretraining_checkpoint_sha256':sha(prepath),\\n             'attention':'bidirectional' if model.bidirectional else 'causal',\\n             'parameters':sum(p.numel() for p in model.parameters()),\\n             'trainable_parameters':sum(p.numel() for p in model.parameters() if p.requires_grad),\\n             'trainable_names':[n for n,p in model.named_parameters() if p.requires_grad],\\n             'frozen_backbone_sha256_before':frozen_before,\\n             'backbone_dropout':'remains active during training in all arms; disabled at evaluation'}\\n    groups=[]\\n    for module,lr in [(model.backbone,c['encoder_lr']),(model.adapters,c['adapter_lr']),(model.head,c['head_lr'])]:\\n        params=[p for p in module.parameters() if p.requires_grad]\\n        if params:groups.append({'params':params,'lr':lr})\\n    opt=torch.optim.AdamW(groups,weight_decay=c['weight_decay'])\\n    y=arrays['train'][1];weight=float((y==0).sum()/(y==1).sum())\\n    criterion=nn.BCEWithLogitsLoss(pos_weight=torch.tensor(weight,device=device))\\n    history=[];best_auc=-1.;best_loss=float('inf');bad=0;start=1\\n    bestpath=folder/'best.pt';lastpath=folder/'last.pt'\\n    if lastpath.exists():\\n        st=loadpt(lastpath,device);model.load_state_dict(st['model']);opt.load_state_dict(st['optimizer'])\\n        history=st['history'];best_auc=st['best_auc'];best_loss=st['best_loss'];bad=st['bad'];start=st['epoch']+1\\n    for epoch in range(start,c['finetune_epochs']+1):\\n        if bad>=c['finetune_patience']:break\\n        seed_all(c['seed']+10000+epoch);model.train();total=0.;n=0;tick=time.time()\\n        for x,yb in loader(arrays['train'],c['batch_size'],True,c['seed']+10000+epoch):\\n            x=x.to(device);yb=yb.to(device);opt.zero_grad(set_to_none=True);loss=criterion(model(x),yb)\\n            if not torch.isfinite(loss):raise RuntimeError('Non-finite fine-tuning loss')\\n            loss.backward();nn.utils.clip_grad_norm_(model.parameters(),1.);opt.step();total+=float(loss.detach())*len(x);n+=len(x)\\n        p,vl=infer(model,arrays['validation'],c['batch_size'],device,criterion)\\n        auc=float(roc_auc_score(arrays['validation'][1],p));ap=float(average_precision_score(arrays['validation'][1],p))\\n        improved=auc>best_auc+1e-9 or (abs(auc-best_auc)<=1e-9 and vl<best_loss-1e-7)\\n        if improved:\\n            best_auc=auc;best_loss=vl;bad=0\\n            savept(bestpath,{'model':model.state_dict(),'cfg':asdict(cfg),'variant':variant,'epoch':epoch,'val_auc':auc,'val_loss':vl})\\n        else:bad+=1\\n        history.append(dict(epoch=epoch,train_loss=total/n,val_loss=vl,val_auroc=auc,val_ap=ap))\\n        savept(lastpath,dict(model=model.state_dict(),optimizer=opt.state_dict(),history=history,best_auc=best_auc,best_loss=best_loss,bad=bad,epoch=epoch))\\n        writecsv(folder/'history.csv',history)\\n        print(f'FT  {variant:8s} {epoch:03d} train={total/n:.5f} val={vl:.5f} AUROC={auc:.4f} AP={ap:.4f} {time.time()-tick:.1f}s',flush=True)\\n    st=loadpt(bestpath,device);model.load_state_dict(st['model']);p,vl=infer(model,arrays['validation'],c['batch_size'],device,criterion)\\n    threshold=choose_threshold(arrays['validation'][1],p)\\n    if model.use_adapter:\\n        profile['frozen_backbone_sha256_after']=backbone_digest(model)\\n        if profile['frozen_backbone_sha256_after']!=frozen_before:raise RuntimeError('Frozen backbone changed')\\n    dump(folder/'training_profile.json',profile)\\n    result={'variant':variant,'best_epoch':st['epoch'],'parameters':sum(p.numel() for p in model.parameters()),'trainable_parameters':profile['trainable_parameters'],'val_loss':vl,**metrics(arrays['validation'][1],p,threshold)}\\n    predictions(folder/'validation_predictions.csv',arrays['validation'],p,threshold)\\n    dump(summarypath,result)\\n    del model,opt\\n    if device.type=='cuda':torch.cuda.empty_cache()\\n    return result\\n\\ndef plot_histories(root):\\n    import matplotlib\\n    matplotlib.use('Agg')\\n    import matplotlib.pyplot as plt\\n    fig,axes=plt.subplots(2,len(VARIANTS),figsize=(15,7),squeeze=False)\\n    for j,v in enumerate(VARIANTS):\\n        rows=list(csv.DictReader((root/v/'history.csv').open(encoding='utf-8-sig')))\\n        e=[int(r['epoch']) for r in rows]\\n        axes[0,j].plot(e,[float(r['train_loss']) for r in rows],label='train weighted BCE')\\n        axes[0,j].plot(e,[float(r['val_loss']) for r in rows],label='validation weighted BCE')\\n        axes[0,j].set_title(v);axes[0,j].legend(fontsize=8)\\n        axes[1,j].plot(e,[float(r['val_auroc']) for r in rows]);axes[1,j].set(xlabel='Epoch',ylabel='Validation AUROC',ylim=(0,1))\\n    fig.tight_layout();fig.savefig(root/'training_curves.png',dpi=160);plt.close(fig)\\n\\ndef compare(data_path,split_path,root,c,device):\\n    root.mkdir(parents=True,exist_ok=True)\\n    configs={v:ModelConfig(length=math.ceil(50001/c['factor']),patch=c['patch'],dim=c['dim'],layers=c['layers'],heads=c['heads'],ff=c['ff'],dropout=c['dropout'],adapter_dim=c['adapter_dim']) for v in VARIANTS}\\n    evidence=dict(config=c,model_configs={v:asdict(cfg) for v,cfg in configs.items()},split_sha256=sha(split_path),source_sha256={n:sha(ROOT/n) for n in ('model.py','data.py','run.py','cv.py','dataset_manifest.json','fixed_folds.json')})\\n    if (root/'protocol.json').exists():\\n        old=read(root/'protocol.json')\\n        if old!=evidence:raise ValueError('이 결과 폴더의 설정/코드/분할이 다릅니다. 새 RUN_NAME을 사용하세요.')\\n    else:dump(root/'protocol.json',evidence)\\n    dump(root/'environment.json',{'python':sys.version,'torch':str(torch.__version__),'numpy':np.__version__,'device':str(device),'gpu':torch.cuda.get_device_name(0) if device.type=='cuda' else None})\\n    arrays=load_roles(data_path,split_path,('train','validation'),c['factor'])\\n    for role,arr in arrays.items():print(role,arr[0].shape,'normal',int((arr[1]==0).sum()),'defect',int((arr[1]==1).sum()),flush=True)\\n    cfg=configs[VARIANTS[0]]\\n    path=pretrain('cnn',cfg,c,arrays,root,device)\\n    results=[finetune(v,path,configs[v],c,arrays,root,device) for v in VARIANTS]\\n    writecsv(root/'validation_comparison.csv',results)\\n    winner=max(results,key=lambda r:(r['auroc'],-r['val_loss']))['variant']\\n    selection={'selected_by_validation':winner,'rule':'max validation AUROC, tie min validation weighted BCE; complete tie CAUSAL_ADAPTER/BI_FULL/BI_ADAPTER order',\\n               'checkpoint_sha256':{v:sha(root/v/'best.pt') for v in VARIANTS},'validation_file_sha256':{v:sha(root/v/'validation.json') for v in VARIANTS},\\n               'note':'Frozen BEFORE test. All 144 records were previously explored; this is internal stratified cross-validation.'}\\n    dump(root/'selection_before_test.json',selection);plot_histories(root)\\n    print('VALIDATION COMPARISON',json.dumps(results,ensure_ascii=False,indent=2),flush=True)\\n    print('Selected on validation:',winner,'| No test waveforms loaded in compare stage.',flush=True)\\n\\ndef auc_ci(y,p,n=3000):\\n    rng=np.random.default_rng(734);a=p[y==0];b=p[y==1]\\n    pair=(b[:,None]>a).astype(float)+.5*(b[:,None]==a)\\n    vals=[pair[np.ix_(rng.integers(len(b),size=len(b)),rng.integers(len(a),size=len(a)))].mean() for _ in range(n)]\\n    return np.quantile(vals,[.025,.975]).tolist()\\n\\ndef final_test(data_path,split_path,root,device):\\n    protocol=read(root/'protocol.json');c=protocol['config'];selection=read(root/'selection_before_test.json')\\n    if sha(split_path)!=protocol['split_sha256']:raise ValueError('Split changed')\\n    for n,h in protocol['source_sha256'].items():\\n        if sha(ROOT/n)!=h:raise ValueError('Code changed after training: '+n)\\n    for v in VARIANTS:\\n        if sha(root/v/'best.pt')!=selection['checkpoint_sha256'][v] or sha(root/v/'validation.json')!=selection['validation_file_sha256'][v]:raise ValueError('Checkpoint or validation selection changed')\\n    if (root/'test_comparison.csv').exists():\\n        print('Already evaluated: reusing saved test results.');print((root/'test_comparison.csv').read_text(encoding='utf-8-sig'));return\\n    # First test decoding occurs here, after all model selection is frozen.\\n    test=load_roles(data_path,split_path,('test',),c['factor'])['test']\\n    results=[];curves={}\\n    for v in VARIANTS:\\n        cfg=ModelConfig(**protocol['model_configs'][v])\\n        model=Classifier(cfg,v).to(device);model.load_state_dict(loadpt(root/v/'best.pt',device)['model'])\\n        threshold=read(root/v/'validation.json')['threshold'];p,_=infer(model,test,c['batch_size'],device)\\n        r={'variant':v,'selected_by_validation':v==selection['selected_by_validation'],**metrics(test[1],p,threshold)}\\n        lo,hi=auc_ci(test[1],p);r['auroc_bootstrap_low']=lo;r['auroc_bootstrap_high']=hi\\n        predictions(root/v/'test_predictions.csv',test,p,threshold);dump(root/v/'test.json',r);results.append(r);curves[v]=p\\n        del model\\n    import matplotlib\\n    matplotlib.use('Agg')\\n    import matplotlib.pyplot as plt\\n    fig,ax=plt.subplots(figsize=(6,5))\\n    for r in results:\\n        fpr,tpr,_=roc_curve(test[1],curves[r['variant']]);ax.plot(fpr,tpr,label=f\\\"{r['variant']}: {r['auroc']:.3f}\\\")\\n    ax.plot([0,1],[0,1],'k--',alpha=.4);ax.legend();ax.set(xlabel='False positive rate',ylabel='True positive rate',title=f\\\"Fold test: {int((test[1]==0).sum())} normal + {int((test[1]==1).sum())} defect\\\")\\n    fig.tight_layout();fig.savefig(root/'test_roc.png',dpi=160);plt.close(fig)\\n    dump(root/'test_notes.json',{'selected_model':selection['selected_by_validation'],'test_counts':{'normal':int((test[1]==0).sum()),'defect':int((test[1]==1).sum())},\\n                              'CI':'Stratified fixed-model bootstrap; few defects per fold; ignores dependence, training variability and previous data exploration.',\\n                              'interpretation':'Compare prespecified arms. Do not select the best test arm or retune using these scores. AUROC >= 0.85 is not guaranteed.'})\\n    writecsv(root/'test_comparison.csv',results)\\n    print('FINAL TEST',json.dumps(results,ensure_ascii=False,indent=2),flush=True)\\n\\ndef main():\\n    p=argparse.ArgumentParser();p.add_argument('stage',choices=['compare','test']);p.add_argument('--data',required=True);p.add_argument('--out',required=True)\\n    p.add_argument('--split',required=True);p.add_argument('--config',default=str(ROOT/'config.json'));p.add_argument('--device',choices=['cpu','cuda'],default='cuda');p.add_argument('--threads',type=int,default=2)\\n    a=p.parse_args();torch.set_num_threads(a.threads);device=device_for(a.device)\\n    if a.stage=='compare':compare(a.data,a.split,Path(a.out),read(a.config),device)\\n    else:final_test(a.data,a.split,Path(a.out),device)\\nif __name__=='__main__':main()\\n\",\"prepare_local_metadata.py\":\"\\\"\\\"\\\"Create private recording identities and CV folds locally from a user-provided ZIP.\\\"\\\"\\\"\\nimport argparse\\nimport hashlib\\nimport io\\nimport json\\nimport zipfile\\nfrom pathlib import Path\\nimport numpy as np\\nfrom sklearn.model_selection import StratifiedKFold, train_test_split\\nfrom data import canonical\\n\\nROOT = Path(__file__).resolve().parent\\n\\ndef build_metadata(data_path, seed=42):\\n    records = []\\n    seen_names, seen_hashes = set(), set()\\n    reference_time = None\\n    with zipfile.ZipFile(data_path) as archive:\\n        names = sorted(\\n            name for name in archive.namelist()\\n            if name.lower().endswith(\\\".dat\\\") and not name.startswith(\\\"__MACOSX/\\\")\\n        )\\n        for name in names:\\n            identifier = canonical(name)\\n            if identifier in seen_names:\\n                raise ValueError(\\\"Duplicate canonical recording name\\\")\\n            seen_names.add(identifier)\\n            array = np.loadtxt(io.BytesIO(archive.read(name)))\\n            if array.shape == (50001, 2):\\n                array = array.T\\n            if array.shape != (2, 50001) or not np.isfinite(array).all():\\n                raise ValueError(\\\"Expected a finite two-column/row 50,001-point DAT recording\\\")\\n            time, voltage = array\\n            if not np.all(np.diff(time) > 0):\\n                raise ValueError(\\\"Nonmonotonic time axis\\\")\\n            if reference_time is None:\\n                reference_time = time\\n            elif not np.allclose(time, reference_time, rtol=0, atol=1e-15):\\n                raise ValueError(\\\"Recording time axes differ\\\")\\n            digest = hashlib.sha256(voltage.astype(np.float32).tobytes()).hexdigest()\\n            if digest in seen_hashes:\\n                raise ValueError(\\\"Duplicate waveform content\\\")\\n            seen_hashes.add(digest)\\n            records.append({\\n                \\\"file\\\": identifier,\\n                \\\"label\\\": int(\\\"##결함/\\\" in identifier),\\n                \\\"waveform_sha256\\\": digest,\\n            })\\n    records.sort(key=lambda row: row[\\\"file\\\"])\\n    labels = np.asarray([row[\\\"label\\\"] for row in records], dtype=np.int64)\\n    counts = np.bincount(labels, minlength=2)\\n    if len(records) != 144 or counts.tolist() != [126, 18]:\\n        raise ValueError(\\\"This experiment requires 126 normal and 18 defect recordings\\\")\\n    indices = np.arange(len(records))\\n    documents = []\\n    outer = StratifiedKFold(n_splits=5, shuffle=True, random_state=seed)\\n    for fold, (development, test) in enumerate(outer.split(indices, labels), 1):\\n        inner_seed = 1000 + fold\\n        train, validation = train_test_split(\\n            development, test_size=0.20, random_state=inner_seed,\\n            stratify=labels[development],\\n        )\\n        roles = {\\\"train\\\": train, \\\"validation\\\": validation, \\\"test\\\": test}\\n        sets = {role: set(ids.tolist()) for role, ids in roles.items()}\\n        if (sets[\\\"train\\\"] & sets[\\\"validation\\\"] or sets[\\\"train\\\"] & sets[\\\"test\\\"]\\n                or sets[\\\"validation\\\"] & sets[\\\"test\\\"]):\\n            raise RuntimeError(\\\"Split overlap\\\")\\n        documents.append({\\n            \\\"fold\\\": fold,\\n            \\\"outer_seed\\\": seed,\\n            \\\"inner_seed\\\": inner_seed,\\n            \\\"unit\\\": \\\"original recording\\\",\\n            \\\"stratification\\\": \\\"label only\\\",\\n            \\\"splits\\\": {\\n                role: [dict(records[int(i)]) for i in ids]\\n                for role, ids in roles.items()\\n            },\\n        })\\n    test_ids = [row[\\\"file\\\"] for doc in documents for row in doc[\\\"splits\\\"][\\\"test\\\"]]\\n    if len(test_ids) != 144 or len(set(test_ids)) != 144:\\n        raise RuntimeError(\\\"Every recording must appear in outer test exactly once\\\")\\n    dataset = {\\\"counts\\\": {\\\"normal\\\": 126, \\\"defect\\\": 18}, \\\"records\\\": records}\\n    return dataset, documents\\n\\ndef prepare(data_path, seed=42):\\n    dataset, folds = build_metadata(data_path, seed)\\n    outputs = {\\n        \\\"dataset_manifest.json\\\": dataset,\\n        \\\"fixed_folds.json\\\": folds,\\n    }\\n    existing = [name for name in outputs if (ROOT / name).exists()]\\n    if existing:\\n        if len(existing) != len(outputs):\\n            raise ValueError(\\\"Incomplete local metadata; use a fresh project directory\\\")\\n        for name, content in outputs.items():\\n            saved = json.loads((ROOT / name).read_text(encoding=\\\"utf-8\\\"))\\n            if saved != content:\\n                raise ValueError(\\\"Existing local metadata differs; use a fresh project directory\\\")\\n        print(\\\"Existing local metadata matches this dataset and split seed.\\\")\\n        return\\n    for name, content in outputs.items():\\n        path = ROOT / name\\n        temporary = path.with_suffix(path.suffix + \\\".tmp\\\")\\n        temporary.write_text(\\n            json.dumps(content, ensure_ascii=False, indent=2),\\n            encoding=\\\"utf-8\\\",\\n        )\\n        temporary.replace(path)\\n    print(\\\"Private recording metadata and five folds generated locally.\\\")\\n    print(\\\"The public repository contains no recording identity lists.\\\")\\n\\nif __name__ == \\\"__main__\\\":\\n    parser = argparse.ArgumentParser()\\n    parser.add_argument(\\\"--data\\\", required=True)\\n    parser.add_argument(\\\"--seed\\\", type=int, default=42)\\n    arguments = parser.parse_args()\\n    prepare(arguments.data, arguments.seed)\\n\"}")
for name, content in SOURCE_FILES.items():
    if Path(name).name != name:
        raise ValueError("Unexpected source path")
    (PROJECT / name).write_text(content, encoding="utf-8")
print("Public source prepared:", PROJECT)
print("Files:", sorted(SOURCE_FILES))


In [ ]:
# Upload the original DAT archive
from google.colab import files
uploaded = files.upload()
candidates = []
for name in uploaded:
    path = Path(name).resolve()
    if path.suffix.lower() == ".zip" and zipfile.is_zipfile(path):
        with zipfile.ZipFile(path) as archive:
            dat_files = [n for n in archive.namelist()
                         if n.lower().endswith(".dat") and not n.startswith("__MACOSX/")]
        if dat_files:
            candidates.append(path)
if len(candidates) != 1:
    raise ValueError("Upload exactly one ZIP containing the original 144 DAT recordings.")
DATA_PATH = candidates[0]
del uploaded
print("Dataset:", DATA_PATH.name)


In [ ]:
# Persistent output and resumption
USE_DRIVE = True
RUN_NAME = "cnn_causal_adapter_cv5_seed42_v1"
THREADS = 2
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    STORE = Path("/content/drive/MyDrive/TimeDART_CNN_CausalAdapter")
else:
    STORE = Path("/content/TimeDART_CNN_CausalAdapter_Output")
STORE.mkdir(parents=True, exist_ok=True)
OUTPUT = STORE / RUN_NAME
OUTPUT.mkdir(parents=True, exist_ok=True)

def execute(*args):
    command = [sys.executable, "-u", *map(str, args)]
    with subprocess.Popen(command, cwd=PROJECT, stdout=subprocess.PIPE,
                          stderr=subprocess.STDOUT, text=True,
                          encoding="utf-8", errors="replace", bufsize=1) as process:
        for line in process.stdout:
            print(line, end="", flush=True)
        status = process.wait()
    if status:
        raise subprocess.CalledProcessError(status, command)

print("Output:", OUTPUT)
print("Target model: CAUSAL_ADAPTER; original comparison arms are retained.")
print("Change RUN_NAME if source/configuration/folds change.")


In [ ]:
# Generate private local metadata, check the structure, then train.
# No outer test inference occurs in this training stage.
execute("prepare_local_metadata.py", "--data", DATA_PATH)
execute("checks.py")
execute("cv.py", "train", "--data", DATA_PATH, "--out", OUTPUT,
        "--device", DEVICE, "--threads", THREADS)


In [ ]:
# Evaluate the already selected checkpoints on the outer test folds
execute("cv.py", "test", "--data", DATA_PATH, "--out", OUTPUT,
        "--device", DEVICE, "--threads", THREADS)


In [ ]:
# Ratios: five-fold mean. Counts: sum over the five test folds.
import pandas as pd
summary = pd.read_csv(OUTPUT / "cv_summary.csv")
fold_metrics = pd.read_csv(OUTPUT / "cv_fold_metrics.csv")
metric_columns = [("PR-AUC (AP)", "average_precision"),
                  ("Defect recall", "recall"), ("BA", "balanced_accuracy"),
                  ("F1", "f1"), ("Specificity", "specificity")]
for _, row in summary.iterrows():
    variant = row["variant"]
    print("\n" + "=" * 65)
    print(variant)
    print("=" * 65)
    print(f"AUROC           : {row['auroc_mean']:.10f} ± {row['auroc_std']:.10f}")
    for label, key in metric_columns:
        print(f"{label:<16}: {row[key + '_mean']:.10f}  (fold mean)")
    group = fold_metrics[fold_metrics["variant"] == variant]
    for label in ("FN", "FP", "TN", "TP"):
        print(f"{label:<16}: {int(group[label.lower()].sum())}  (fold total)")
from IPython.display import display, Image
display(Image(filename=str(OUTPUT / "cv_auroc.png")))


In [ ]:
# Download outputs for private local use. Do not commit generated result archives.
RESULT_ZIP = STORE / (RUN_NAME + "_results.zip")
with zipfile.ZipFile(RESULT_ZIP, "w", zipfile.ZIP_DEFLATED) as archive:
    for name in SOURCE_FILES:
        archive.write(PROJECT / name, "source/" + name)
    for name in ("dataset_manifest.json", "fixed_folds.json"):
        archive.write(PROJECT / name, "private_metadata/" + name)
    for path in OUTPUT.rglob("*"):
        if path.is_file() and "__pycache__" not in path.parts:
            archive.write(path, "results/" + str(path.relative_to(OUTPUT)))
print("Private result archive:", RESULT_ZIP)
files.download(str(RESULT_ZIP))
